# CS336 Spring 2026 - Toy BPE Trainer
----
> 目的是实现教学版 Toy BPE Trainer.

先给出 Toy BPE Trainer 的训练循环:
1. UTF-8 bytes 作为基础 token
2. 统计 adjacent pair frequencies
3. 选择频率最高的 pair
4. 合并所有不重叠出现(merge)
5. 更新 vocabulary
6. 重复训练若干轮

## 1. 从 UTF-8 bytes 开始
对于 ASCII 文本, byte 的值刚好与字符编码对应, 但后续仍坚持从 `text.encode("utf-8")` 开始.

In [1]:
text = "the cat in the hat"

byte_values = list(text.encode("utf-8"))

print("=====UTF-8 byte values=====")
print(byte_values)

print("=====UTF-8 byte values as string=====")
print(bytes(byte_values))

=====UTF-8 byte values=====
[116, 104, 101, 32, 99, 97, 116, 32, 105, 110, 32, 116, 104, 101, 32, 104, 97, 116]
=====UTF-8 byte values as string=====
b'the cat in the hat'


## 2. 将 token 表示为 bytes
正式实现时可以使用 token IDs. 这里暂时使用 bytes, 可以让每轮 merge 特别直观.

例如 `"the"` 并不是一个 token, 而是 `b"t" b"h" b"e"` 三个 byte tokens.

In [2]:
tokens = [bytes([b]) for b in text.encode("utf-8")]

print("tokens = ", tokens)

tokens =  [b't', b'h', b'e', b' ', b'c', b'a', b't', b' ', b'i', b'n', b' ', b't', b'h', b'e', b' ', b'h', b'a', b't']


## 3. 统计相邻的 pair
长度为 N 的 token sequence, 一共有 N-1 个 adjacent pairs. 例如
```text
[a, b, a, b]
    ↓
pairs: (a, b) (b, a) (a, b)
    ↓
(a, b) -> 2
(b, a) -> 1
```

In [3]:
from collections import Counter


def count_pairs(tokens: list[bytes]) -> Counter[tuple[bytes, bytes]]:
    # 统计相邻的 token 对

    return Counter(zip(tokens, tokens[1:]))


example = [b"a", b"b", b"a", b"b"]
print("number of adjacent pairs = ", count_pairs(example))

number of adjacent pairs =  Counter({(b'a', b'b'): 2, (b'b', b'a'): 1})


上述函数还可以写为
```Python 
from collections import Counter
from itertools import pairwise

   def count_pairs(tokens: list[bytes]) -> Counter[tuple[bytes, bytes]]:
       return Counter(pairwise(tokens))
```
这是在 Python 3.10 及以上版本中标准库的写法, 与上述代码等价. 

**为什么 `zip(tokens, tokens[1:])` 有效?**

```text
tokens = [a, b, c, d]
    ↓
tokens = a b c d
tokens[1:] = b c d
    ↓
zip(tokens, tokens[1:]) = [(a, b), (b, c), (c, d)]
```
此操作在统计相邻对时经常使用.

## 4. 检查 overlapping pair 的统计

In [4]:
example = [b"a", b"a", b"a", b"a"]

print(count_pairs(example))

Counter({(b'a', b'a'): 3})


上述结果表明: pair frequency 可以包含 overlapping occurrences.

## 5. 实现 merge
现在已经统计了所有的 adjacent pair, 知道了哪个 pair 的频率最高. 下一步就是如何将这个 pair 合并成一个 token. 例如:
```text
[a, b, a, b, c]
    ↓
选择 pair: (a, b)
    ↓
[ab, ab, c]
```
但是要考虑 overlapping 问题. 例如:
```text
[a, a, a]
    ↓
选择 pair: (a, a)
    ↓
不能得到: [aa, aa], 而是得到: [aa, a]
```
> merge replacement 必须是 non-overlapping 的.

In [5]:
# 实现 merge 函数
def merge_pair(tokens: list[bytes], pair: tuple[bytes, bytes]) -> list[bytes]:
    # 将 pair 中的两个字节合并成一个字节
    left, right = pair
    merged_token = left + right

    output: list[bytes] = []
    i = 0

    while i < len(tokens):
        if i + 1 < len(tokens) and tokens[i] == left and tokens[i + 1] == right:
            output.append(merged_token)
            i += 2  # 考虑 overlapping pairs
        else:
            output.append(tokens[i])
            i += 1

    return output

In [6]:
print(merge_pair([b"a", b"a", b"a"], (b"a", b"a")))

print(merge_pair([b"a", b"a", b"a", b"a"], (b"a", b"a")))

[b'aa', b'a']
[b'aa', b'aa']


## 6. 一个重要的区别
对于 `a a a a` , pair counting: 
$$
count(a, a) = 3
$$
但是执行 merge 操作后, 实际执行此处只有 2. 所以
$$
\text{pair frequency}\neq \text{number of replacements}
$$

## 7. Pair 的选择规则
Toy BPE 每轮都会统计所有的 pair 并且选择 frequency 最大的pair. 但是若两个 pair 的频率相等, 则必须使用 **deterministic tie-breaking** 规则来选择 pair. 

> Toy BPE 会优先比较 frequency, 若 frequency 相等, 则会比较 pair 的字典顺序, 选择字典顺序更小的 pair.

In [7]:
def select_pair(pair_counts: Counter[tuple[bytes, bytes]]) -> tuple[bytes, bytes]:
    # 在频率相同时, 选择 Pair

    return max(pair_counts, key=lambda pair: (pair_counts[pair], pair))

In [8]:
pair_counts = count_pairs(tokens)

for pair, frequency in pair_counts.most_common():
    print(pair, frequency)

print("selected:", select_pair(pair_counts))

(b't', b'h') 2
(b'h', b'e') 2
(b'e', b' ') 2
(b'a', b't') 2
(b' ', b'c') 1
(b'c', b'a') 1
(b't', b' ') 1
(b' ', b'i') 1
(b'i', b'n') 1
(b'n', b' ') 1
(b' ', b't') 1
(b' ', b'h') 1
(b'h', b'a') 1
selected: (b't', b'h')


## 8. 第一轮 BPE merge
对于初始文本
```text
t | h | e | | c | a | t | | i| n | | t | h | e | | h | a | t
    ↓
选择 (b"t", b"h")
    ↓
构造 b"t" + b"h" = b"th"
    ↓
替换 th
    ↓
th | e | | c | a | t | | i| n | | th | e | | h | a | t
```

## 9. Vocabulary 增长
1. **初始化 vocabulary**:
    ```python
    vocab = {i:bytes([i]) for i in range(256)}
    ```
    即所有的 token IDs 在 0-255 之间, 对应单 byte.

2. **第一轮 merge**:
    第一轮合并的是 `b"t" + b"h"`, 所以在 vocabulary 中增加
    ```python
    vocab[256] = b"th"
    ```

3. **第二轮 merge**:
    若第二轮合并 `b"th" + b"e"`, 则要在 vocabulary 中增加
    ```python
    vocab[257] = b"the"
    ```
4. **重复此操作**.
    > 所以 BPE vocabulary 是逐步构造出来的.

## 10. 为训练轨迹写辅助函数
在阅读时, 若空格没有显示, 为了方便阅读, 可以将空格显示为
```text
␠
```
例如, `b"the "` 会显示为 `b"the␠"`, 可以方便后续阅读

In [9]:
def readable(tokens: list[bytes]) -> str:
    # 将空格替换为 ␠

    return " | ".join(
        token.decode(
            "utf-8",
            errors="replace",
        ).replace(" ", "␠")
        for token in tokens
    )


## 11. Toy BPE Training Loop
现在将上述内容连接起来:
```text
text
    ↓
UTF-8
    ↓
single byte tokens
    ↓
count pairs
    ↓
select pairs
    ↓
create vocabulary entry
    ↓
merge
    ↓
repeat
```

In [10]:
def train_toy_bpe(text: str, number_merges: int):
    # 完整实现 Toy BPE Trainer
    tokens = [bytes([b]) for b in text.encode("utf-8")]
    vocab: dict[int, bytes] = {i: bytes([i]) for i in range(256)}
    merges: list[tuple[bytes, bytes]] = []

    print("=====初始化=====")
    print(readable(tokens))

    for step in range(number_merges):
        pair_counts = count_pairs(tokens)

        if not pair_counts:
            break

        pair = select_pair(pair_counts)
        frequency = pair_counts[pair]

        new_token = pair[0] + pair[1]
        new_token_id = len(vocab)

        vocab[new_token_id] = new_token
        merges.append(pair)

        tokens = merge_pair(tokens, pair)

        print(f"=====第{step + 1}步合并=====")
        print("selected pair = ", pair)
        print("frequency = ", frequency)
        print("new token = ", new_token)
        print("new token id = ", new_token_id)
        print("sequence = ", readable(tokens))

    return vocab, merges, tokens

In [11]:
# 运行3次merge操作，得到最终的token列表
vocab, merges, final_tokens = train_toy_bpe("the cat in the hat", number_merges=3)

=====初始化=====
t | h | e | ␠ | c | a | t | ␠ | i | n | ␠ | t | h | e | ␠ | h | a | t
=====第1步合并=====
selected pair =  (b't', b'h')
frequency =  2
new token =  b'th'
new token id =  256
sequence =  th | e | ␠ | c | a | t | ␠ | i | n | ␠ | th | e | ␠ | h | a | t
=====第2步合并=====
selected pair =  (b'th', b'e')
frequency =  2
new token =  b'the'
new token id =  257
sequence =  the | ␠ | c | a | t | ␠ | i | n | ␠ | the | ␠ | h | a | t
=====第3步合并=====
selected pair =  (b'the', b' ')
frequency =  2
new token =  b'the '
new token id =  258
sequence =  the␠ | c | a | t | ␠ | i | n | ␠ | the␠ | h | a | t


In [12]:
# 查看 merge
print("merges:")

for i, pair in enumerate(merges):
    print(i, pair)


merges:
0 (b't', b'h')
1 (b'th', b'e')
2 (b'the', b' ')


In [13]:
# 查看新的 vocabulary
print("new vocab entries:")

for token_id in range(256, 256 + len(merges)):
    print(token_id, vocab[token_id])


new vocab entries:
256 b'th'
257 b'the'
258 b'the '


## 12. Vocabulary 的意义
基础词表为:
```text
0     -> b"\x00"
1     -> b"\x01"
...
97    -> b"a"
...
255   -> b"\xff"
```
在 merge 后新增:
```text
256 -> b"th"
257 -> b"the"
258 -> b"the "
```
所以

<center>
token IDs -> byte sequence
</center>

一个 token 不再对应一个 byte.

In [14]:
# 本节最后验证测试
assert count_pairs([b"a", b"b", b"a", b"b"])[(b"a", b"b")] == 2
assert count_pairs([b"a", b"b", b"a", b"b"])[(b"b", b"a")] == 1
assert count_pairs([b"a", b"a", b"a", b"a"])[(b"a", b"a")] == 3
print("Count pairs test passed!")

assert merge_pair([b"a", b"a", b"a"], (b"a", b"a")) == [b"aa", b"a"]
assert merge_pair([b"a", b"a", b"a", b"a"], (b"a", b"a")) == [b"aa", b"aa"]
print("Merge pair test passed!")

assert merges == [(b"t", b"h"), (b"th", b"e"), (b"the", b" ")]
print("Merges test passed!")

assert vocab[256] == b"th"
assert vocab[257] == b"the"
assert vocab[258] == b"the "
print("Vocab test passed!")

print("All tests passed!")

Count pairs test passed!
Merge pair test passed!
Merges test passed!
Vocab test passed!
All tests passed!


## 13. Training 和 Encoding 不要混淆
现在做的是 Training:
```text
corpus
    ↓
统计 frequency
    ↓
学习 merge
    ↓
得到 vocabulary
```
后续做 Encoding:
```text
new text
    ↓
已经学好的 merges + vocabulary
    ↓
得到 token IDs
```

## 14. 目前 Toy BPE 的局限
> 目前的 Toy BPE 完全不知道什么叫做 tokenization boundary

我们后续还可能学习 `b"the c"`, `b"hat the"` 或者跨越我们不希望跨越的文本结构. 这就是下一节 **Pre-tokenization** 索要解决的问题.

In [15]:
# 新增测试实验一
train_toy_bpe("aaaa", number_merges=3)

=====初始化=====
a | a | a | a
=====第1步合并=====
selected pair =  (b'a', b'a')
frequency =  3
new token =  b'aa'
new token id =  256
sequence =  aa | aa
=====第2步合并=====
selected pair =  (b'aa', b'aa')
frequency =  1
new token =  b'aaaa'
new token id =  257
sequence =  aaaa


({0: b'\x00',
  1: b'\x01',
  2: b'\x02',
  3: b'\x03',
  4: b'\x04',
  5: b'\x05',
  6: b'\x06',
  7: b'\x07',
  8: b'\x08',
  9: b'\t',
  10: b'\n',
  11: b'\x0b',
  12: b'\x0c',
  13: b'\r',
  14: b'\x0e',
  15: b'\x0f',
  16: b'\x10',
  17: b'\x11',
  18: b'\x12',
  19: b'\x13',
  20: b'\x14',
  21: b'\x15',
  22: b'\x16',
  23: b'\x17',
  24: b'\x18',
  25: b'\x19',
  26: b'\x1a',
  27: b'\x1b',
  28: b'\x1c',
  29: b'\x1d',
  30: b'\x1e',
  31: b'\x1f',
  32: b' ',
  33: b'!',
  34: b'"',
  35: b'#',
  36: b'$',
  37: b'%',
  38: b'&',
  39: b"'",
  40: b'(',
  41: b')',
  42: b'*',
  43: b'+',
  44: b',',
  45: b'-',
  46: b'.',
  47: b'/',
  48: b'0',
  49: b'1',
  50: b'2',
  51: b'3',
  52: b'4',
  53: b'5',
  54: b'6',
  55: b'7',
  56: b'8',
  57: b'9',
  58: b':',
  59: b';',
  60: b'<',
  61: b'=',
  62: b'>',
  63: b'?',
  64: b'@',
  65: b'A',
  66: b'B',
  67: b'C',
  68: b'D',
  69: b'E',
  70: b'F',
  71: b'G',
  72: b'H',
  73: b'I',
  74: b'J',
  75: b'K',
  76: b'

In [16]:
# 新增测试实现二
train_toy_bpe("abababab", number_merges=3)

=====初始化=====
a | b | a | b | a | b | a | b
=====第1步合并=====
selected pair =  (b'a', b'b')
frequency =  4
new token =  b'ab'
new token id =  256
sequence =  ab | ab | ab | ab
=====第2步合并=====
selected pair =  (b'ab', b'ab')
frequency =  3
new token =  b'abab'
new token id =  257
sequence =  abab | abab
=====第3步合并=====
selected pair =  (b'abab', b'abab')
frequency =  1
new token =  b'abababab'
new token id =  258
sequence =  abababab


({0: b'\x00',
  1: b'\x01',
  2: b'\x02',
  3: b'\x03',
  4: b'\x04',
  5: b'\x05',
  6: b'\x06',
  7: b'\x07',
  8: b'\x08',
  9: b'\t',
  10: b'\n',
  11: b'\x0b',
  12: b'\x0c',
  13: b'\r',
  14: b'\x0e',
  15: b'\x0f',
  16: b'\x10',
  17: b'\x11',
  18: b'\x12',
  19: b'\x13',
  20: b'\x14',
  21: b'\x15',
  22: b'\x16',
  23: b'\x17',
  24: b'\x18',
  25: b'\x19',
  26: b'\x1a',
  27: b'\x1b',
  28: b'\x1c',
  29: b'\x1d',
  30: b'\x1e',
  31: b'\x1f',
  32: b' ',
  33: b'!',
  34: b'"',
  35: b'#',
  36: b'$',
  37: b'%',
  38: b'&',
  39: b"'",
  40: b'(',
  41: b')',
  42: b'*',
  43: b'+',
  44: b',',
  45: b'-',
  46: b'.',
  47: b'/',
  48: b'0',
  49: b'1',
  50: b'2',
  51: b'3',
  52: b'4',
  53: b'5',
  54: b'6',
  55: b'7',
  56: b'8',
  57: b'9',
  58: b':',
  59: b';',
  60: b'<',
  61: b'=',
  62: b'>',
  63: b'?',
  64: b'@',
  65: b'A',
  66: b'B',
  67: b'C',
  68: b'D',
  69: b'E',
  70: b'F',
  71: b'G',
  72: b'H',
  73: b'I',
  74: b'J',
  75: b'K',
  76: b'

In [17]:
# 新增测试实验三
train_toy_bpe("banana banana", number_merges=5)

=====初始化=====
b | a | n | a | n | a | ␠ | b | a | n | a | n | a
=====第1步合并=====
selected pair =  (b'n', b'a')
frequency =  4
new token =  b'na'
new token id =  256
sequence =  b | a | na | na | ␠ | b | a | na | na
=====第2步合并=====
selected pair =  (b'na', b'na')
frequency =  2
new token =  b'nana'
new token id =  257
sequence =  b | a | nana | ␠ | b | a | nana
=====第3步合并=====
selected pair =  (b'b', b'a')
frequency =  2
new token =  b'ba'
new token id =  258
sequence =  ba | nana | ␠ | ba | nana
=====第4步合并=====
selected pair =  (b'ba', b'nana')
frequency =  2
new token =  b'banana'
new token id =  259
sequence =  banana | ␠ | banana
=====第5步合并=====
selected pair =  (b'banana', b' ')
frequency =  1
new token =  b'banana '
new token id =  260
sequence =  banana␠ | banana


({0: b'\x00',
  1: b'\x01',
  2: b'\x02',
  3: b'\x03',
  4: b'\x04',
  5: b'\x05',
  6: b'\x06',
  7: b'\x07',
  8: b'\x08',
  9: b'\t',
  10: b'\n',
  11: b'\x0b',
  12: b'\x0c',
  13: b'\r',
  14: b'\x0e',
  15: b'\x0f',
  16: b'\x10',
  17: b'\x11',
  18: b'\x12',
  19: b'\x13',
  20: b'\x14',
  21: b'\x15',
  22: b'\x16',
  23: b'\x17',
  24: b'\x18',
  25: b'\x19',
  26: b'\x1a',
  27: b'\x1b',
  28: b'\x1c',
  29: b'\x1d',
  30: b'\x1e',
  31: b'\x1f',
  32: b' ',
  33: b'!',
  34: b'"',
  35: b'#',
  36: b'$',
  37: b'%',
  38: b'&',
  39: b"'",
  40: b'(',
  41: b')',
  42: b'*',
  43: b'+',
  44: b',',
  45: b'-',
  46: b'.',
  47: b'/',
  48: b'0',
  49: b'1',
  50: b'2',
  51: b'3',
  52: b'4',
  53: b'5',
  54: b'6',
  55: b'7',
  56: b'8',
  57: b'9',
  58: b':',
  59: b';',
  60: b'<',
  61: b'=',
  62: b'>',
  63: b'?',
  64: b'@',
  65: b'A',
  66: b'B',
  67: b'C',
  68: b'D',
  69: b'E',
  70: b'F',
  71: b'G',
  72: b'H',
  73: b'I',
  74: b'J',
  75: b'K',
  76: b'